In [1]:
%pip install rpy2


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os.environ['R_HOME'] = r'C:\Program Files\R\   -4.4.1'  


In [3]:
import rpy2
import rpy2.robjects as ro

# Check if R can be accessed
try:
    print(ro.r('R.version.string'))
except Exception as e:
    print(f"Error: {e}")


OSError: cannot load library 'C:\Program Files\R\R-4.4.1\bin\x64\R.dll': error 0x7e

In [ ]:
# Load the rpy2 extension for Jupyter Notebook
%load_ext rpy2.ipython

# Start the R code block using the %%R magic command
%%R

# Install and load required libraries
if (!require(pdftools)) install.packages("pdftools")
if (!require(tesseract)) install.packages("tesseract")
if (!require(stringr)) install.packages("stringr")
library(pdftools)
library(tesseract)
library(stringr)

# Define paths
pdf_folder <- "C:/Users/nici_/OneDrive/Desktop/Masterarbeit/Factsheets"
output_text_folder <- "C:/Users/nici_/OneDrive/Desktop/Masterarbeit/Factsheets/Text"
output_tables_folder <- "C:/Users/nici_/OneDrive/Desktop/Masterarbeit/Factsheets/Tables"

# Ensure output directories exist
dir.create(output_text_folder, showWarnings = FALSE, recursive = TRUE)
dir.create(output_tables_folder, showWarnings = FALSE, recursive = TRUE)

# Define company names to be anonymized
company_names <- c("Vontobel", "Nomura", "iShares", "Stiftung für den flexiblen Altersrücktritt im Bauhauptgewerbe", "VAM", "AssetManagement")

# Initialize a list for tracking unique company names with an index
company_index <- list()
company_count <- 0

anonymize_text <- function(text) {
  text <- vapply(text, function(line) {
    # Anonymize emails
    line <- str_replace_all(line, "\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}\\b", "[EMAIL]")
    
    # Anonymize names (capitalized first and last names)
    line <- str_replace_all(line, "\\b([A-Z][a-z]+\\s[A-Z][a-z]+)\\b", "[NAME]")
    
    # Anonymize addresses, with improved matching for compound street names
    address_pattern <- "(?i)\\b\\d{1,5}\\s+([A-Z][a-z]*\\s){0,4}([A-Z][a-z]+)?(Strasse|strasse|Platz|platz|Weg|weg|Gasse|gasse|Allee|allee|Chaussee|chaussee|Ring|ring|Chemin|chemin|Rue|rue|Route|route|Cours|cours|Place|place|Allée|allée|Impasse|impasse|Voie|voie|strasse|gasse|platz|allee|weg|ring)\\b"
    line <- str_replace_all(line, address_pattern, "[ADDRESS]")
    
    # Anonymize Swiss phone numbers
    line <- str_replace_all(line, "\\+41\\s?\\d{2}\\s?\\d{3}\\s?\\d{2}\\s?\\d{2}", "[PHONE]")
    
    # Anonymize Swiss postcodes (4-digit numbers starting with 3-9)
    line <- str_replace_all(line, "\\b[3-9]\\d{3}\\b", "[POSTCODE]")
    
    # Anonymize and index company names
    for (company in company_names) {
      pattern <- paste0("\\b(?i)", company, "\\b")
      if (grepl(pattern, line, perl = TRUE)) {
        if (!company %in% names(company_index)) {
          company_count <<- company_count + 1
          company_index[[company]] <<- paste0("[COMPANY_", company_count, "]")
        }
        line <- str_replace_all(line, pattern, company_index[[company]])
      }
    }
    return(line)
  }, character(1))
  
  return(text)
}

# Function to anonymize company names in filenames
anonymize_filename <- function(filename) {
  for (company in company_names) {
    pattern <- paste0("(?i)", company)
    if (grepl(pattern, filename)) {
      if (!company %in% names(company_index)) {
        company_count <<- company_count + 1
        company_index[[company]] <<- paste0("[COMPANY_", company_count, "]")
      }
      filename <- str_replace_all(filename, pattern, company_index[[company]])
    }
  }
  return(filename)
}

# Function for OCR extraction with fallback to OCR if regular text extraction fails
extract_text_with_fallback <- function(pdf_path) {
  text <- tryCatch({
    pdf_text(pdf_path)
  }, error = function(e) {
    message("Regular text extraction failed. Switching to OCR for: ", pdf_path)
    return(NULL)
  })
  
  if (is.null(text) || length(text) == 0 || all(is.na(text))) {
    pages <- pdf_convert(pdf_path, dpi = 300)
    text <- sapply(pages, function(img) tesseract::ocr(img))
    unlink(pages)  # Clean up temporary images
  }
  
  return(text)
}

# Function to detect table-like data from text lines based on numeric patterns
extract_table_like_rows <- function(text_lines) {
  # Detect lines that resemble tables based on numeric and spacing patterns
  table_lines <- text_lines[grepl("^(.*?\\d+.*?\\s{2,}.*?\\d+.*)+$", text_lines)]
  
  if (length(table_lines) > 0) {
    message("Detected potential table rows:")
    print(table_lines)  # Print for debugging purposes
    
    # Split each line by multiple spaces to create columns
    table_data <- strsplit(table_lines, "\\s{2,}")
    
    # Print the initial structure for debugging
    message("Initial structure of `table_data`:")
    str(table_data)
    
    # Filter and standardize `table_data`
    table_data <- lapply(table_data, function(row) {
      # Ensure the row is a valid character vector with non-zero length
      if (is.character(row) && length(row) > 0) {
        return(row)
      } else {
        return(NULL)
      }
    })
    
    # Remove any NULL or empty entries
    table_data <- Filter(function(x) !is.null(x) && length(x) > 0, table_data)
    
    # Check if there is valid data to process
    if (length(table_data) == 0) {
      message("No valid table rows to process.")
      return(NULL)
    }
    
    # Determine the maximum number of columns for standardization
    max_columns <- max(sapply(table_data, length))
    table_data <- lapply(table_data, function(row) {
      # Ensure each row has a consistent number of columns
      length(row) <- max_columns
      row[is.na(row)] <- ""  # Replace NAs with empty strings
      return(row)
    })
    
    # Safely create a data frame and handle errors
    table_df <- tryCatch({
      as.data.frame(do.call(rbind, table_data), stringsAsFactors = FALSE)
    }, error = function(e) {
      message("Error creating data frame from table rows: ", e)
      return(NULL)
    })
    
    return(table_df)
  } else {
    message("No table-like rows detected.")
    return(NULL)
  }
}

# Main processing function to extract and save both text and tables from PDFs
process_pdfs <- function(pdf_folder, output_text_folder, output_tables_folder) {
  pdf_files <- list.files(pdf_folder, pattern = "\\.pdf$", full.names = TRUE)
  
  for (pdf_path in pdf_files) {
    # Use the full filename (without .pdf) for anonymized export
    pdf_name <- tools::file_path_sans_ext(basename(pdf_path))
    pdf_name <- anonymize_filename(pdf_name)
    
    message("Processing PDF: ", pdf_name)
    
    # Extract text from each page with OCR fallback
    text_pages <- extract_text_with_fallback(pdf_path)
    
    # Process each page individually for both text and tables
    for (page_num in seq_along(text_pages)) {
      page_text <- text_pages[[page_num]]
      text_lines <- unlist(strsplit(page_text, "\n"))
      
      # Anonymize each line of text
      anonymized_text <- anonymize_text(text_lines)
      
      # Save anonymized text with left alignment if there is content
      if (any(nzchar(trimws(anonymized_text)))) {
        # Apply left alignment before writing to the file
        left_aligned_text <- trimws(anonymized_text, which = "left")
        txt_file <- file.path(output_text_folder, paste0(pdf_name, "_page_", page_num, ".txt"))
        writeLines(left_aligned_text, txt_file)
        message("Anonymized text saved to: ", txt_file)
      }
      
      # Extract table-like data and save if found
      table_df <- extract_table_like_rows(text_lines)
      if (!is.null(table_df)) {
        # Apply anonymization to each cell in the table
        table_df[] <- lapply(table_df, anonymize_text)
        
        # Save detected table as CSV
        table_file <- file.path(output_tables_folder, paste0(pdf_name, "_page_", page_num, "_table.csv"))
        write.csv(table_df, table_file, row.names = FALSE)
        message("Table saved to: ", table_file)
      } else {
        message("No tables found on page: ", page_num)
      }
    }
  }
}

# Run the function on all PDFs
process_pdfs(pdf_folder, output_text_folder, output_tables_folder)
